# Use viewmastR with large query objects

A query of millions of cells does not need to fit in memory. Open it on disk
(`backed="r"`), train the model once, and let `viewmastR_infer()` read and label the query
in row chunks. This tutorial follows the R article
[How to use viewmastR with large query objects](https://furlan-lab.github.io/viewmastR/articles/BigQuery.html).

| Role | Dataset |
|---|---|
| Reference | healthy bone marrow atlas (Granja et al. 2019, "Greenleaf reference") |
| Query | a large query of your own, raw counts in h5ad (local file; set `large_query` in `data_paths.json`) |

In [ ]:
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import scanpy as sc
import viewmastr as vm
import tutorial_data as td

cfg = td.config()
label = cfg.get("greenleaf_label", "celltype")
ref = td.greenleaf_reference()
query = td.large_query(backed=True)
if cfg.get("large_query_gene_column"):
    query.var_names = query.var[cfg["large_query_gene_column"]].astype(str).to_numpy()
    query.var_names_make_unique(join=".")
print(f"query: {query.n_obs:,} cells x {query.n_vars:,} genes, on disk")

## Train once

Feature selection on the reference keeps the example light; any gene list works. Only
genes present in the query are kept.

In [ ]:
vm.calculate_feature_dispersion(ref)
vm.select_features(ref, top_n=10_000, logmean_ul=-1, logmean_ll=-8)
genes = set(query.var_names)
vg = [g for g in vm.get_selected_features(ref) if g in genes]
model_dir = Path(tempfile.mkdtemp()) / "model"
res = vm.viewmastR(None, ref, label, selected_features=vg, max_epochs=4, train_only=True, model_dir=model_dir)
print(f"{len(vg)} features; training took {res.seconds:.0f} s")

## Infer in chunks

`chunk_size` sets how many cells are read and labelled at a time. It bounds memory; it
does not change the result. Two chunk sizes give identical labels.

In [ ]:
timing = {}
for col, size in (("test1", 20_000), ("test2", 200_000)):
    t0 = time.time()
    vm.viewmastR_infer(query, model_dir, vg, query_celldata_col=col, chunk_size=size)
    timing[size] = time.time() - t0
    print(f"chunk_size={size:,}: {timing[size]:.0f} s")
same = (query.obs["test1"].astype(str) == query.obs["test2"].astype(str)).all()
print("identical labels:", same)

In [ ]:
vm.confusion_matrix(query.obs["test1"], query.obs["test2"])
plt.show()

## Predicted composition

In [ ]:
frac = query.obs["test1"].value_counts(normalize=True).sort_values()
frac.plot.barh(figsize=(5, 4), color="steelblue")
plt.xlabel("Share of cells")
plt.show()

## Versions

In [ ]:
from importlib.metadata import version
for pkg in ("viewmastr", "scanpy", "anndata", "numpy", "scipy"):
    print(f"{pkg:10s} {version(pkg)}")